# Résultats du benchmark

Ce notebook regroupe les résultats des différentes expérimentations 
(baseline, PyCaret, AutoGluon) et les exporte sous forme de tableaux CSV 
dans le dossier `docs/benchmark/results_benchmark/`.

## Charger et fusionner les resultats d'experimentation

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# 1. Définir la structure officielle du benchmark
# ============================================================

benchmark_columns = [
    "solution",
    "horizon_days",
    "model",
    "evaluation_set",
    "evaluation_status",
    "mae",
    "rmse",
    "r2",
    "training_time_seconds",
    "inference_time_seconds",
    "inference_time_per_row_ms",
    "peak_cpu_percent",
    "peak_ram_mb",
    "software_license_cost_eur",
    "selection_decision",
    "notes"
]

# ============================================================
# 2. Charger les résultats des expérimentations
# ============================================================

project_root = Path.cwd().parent

results_dir = (
    project_root / "docs" / "benchmark" / "results_benchmark"
)

pycaret = pd.read_csv(
    results_dir / "pycaret_test_results.csv"
)

autogluon = pd.read_csv(
    results_dir / "autogluon_test_results.csv"
)

baseline = pd.read_csv(
    results_dir / "baseline_test_results.csv"
)

# ============================================================
# 3. Fusionner les trois sources
# ============================================================

sources = pd.concat(
    [pycaret, autogluon, baseline],
    ignore_index=True
)

# Harmoniser les noms des colonnes
measured = sources.rename(columns={
    "tool": "solution",
    "MAE": "mae",
    "RMSE": "rmse",
    "R2": "r2",
    "peak_ram_used_mb": "peak_ram_mb"
}).copy()

# Convertir les horizons : "7d" -> 7, "30d" -> 30
measured["horizon_days"] = (
    measured["horizon"]
    .str.replace("d", "", regex=False)
    .astype(int)
)

# ============================================================
# 4. Compléter les informations du benchmark
# ============================================================

measured["evaluation_set"] = "test"

measured["evaluation_status"] = measured["solution"].map({
    "PyCaret": "POC réalisé",
    "AutoGluon": "POC réalisé",
    "Baseline": "Référence évaluée"
})

# Temps moyen d'inférence sur les 590 observations de test
measured["inference_time_per_row_ms"] = (
    measured["inference_time_seconds"] * 1000 / 590
)

# Coûts de licence des solutions locales
measured["software_license_cost_eur"] = 0.0

# Décisions technologiques
measured["selection_decision"] = measured["solution"].map({
    "PyCaret": "Non retenu",
    "AutoGluon": "Retenu pour validation applicative",
    "Baseline": "Référence métier"
})

# Notes méthodologiques
measured["notes"] = measured["solution"].map({
    "PyCaret": (
        "Entraînement et ressources mesurés globalement "
        "par horizon."
    ),
    "AutoGluon": (
        "Entraînement et ressources mesurés globalement "
        "par horizon."
    ),
    "Baseline": (
        "Modèle de persistance de la volatilité historique ; "
        "aucun entraînement."
    )
})

# ============================================================
# 5. Ajouter les solutions étudiées mais non expérimentées
# ============================================================

not_tested = pd.DataFrame([
    {
        "solution": "H2O AutoML",
        "evaluation_status": "Non expérimenté",
        "selection_decision": "Écarté",
        "notes": (
            "Contraintes techniques liées à Java "
            "et aux dépendances Python."
        )
    },
    {
        "solution": "Azure Automated ML",
        "evaluation_status": "Étudié, non expérimenté",
        "selection_decision": "Non retenu pour le POC local",
        "notes": (
            "Solution cloud étudiée ; "
            "aucune expérimentation réalisée."
        )
    },
    {
        "solution": "Vertex AI",
        "evaluation_status": "Étudié, non expérimenté",
        "selection_decision": "Non testé",
        "notes": (
            "Expérimentation conditionnée au budget cloud."
        )
    }
])

# ============================================================
# 6. Construire le tableau final
# ============================================================

benchmark_results = pd.concat(
    [measured, not_tested],
    ignore_index=True
)

# Conserver les 16 colonnes et leur ordre
benchmark_results = benchmark_results.reindex(
    columns=benchmark_columns
)

# ============================================================
# 7. Vérifier la cohérence des données
# ============================================================

assert benchmark_results.shape == (17, 16)

assert benchmark_results[
    ["solution", "evaluation_status", "selection_decision"]
].notna().all().all()

assert benchmark_results.loc[
    benchmark_results["evaluation_set"] == "test",
    ["horizon_days", "model", "mae", "rmse", "r2"]
].notna().all().all()

assert not benchmark_results.duplicated(
    subset=[
        "solution",
        "horizon_days",
        "model",
        "evaluation_set"
    ]
).any()

# ============================================================
# 8. Exporter le fichier CSV
# ============================================================

export_path = results_dir / "benchmark_results.csv"

benchmark_results.to_csv(
    export_path,
    index=False,
    na_rep="-"
)

# ============================================================
# 9. Afficher le résultat
# ============================================================

print("Export terminé :", export_path)
print("Dimensions :", benchmark_results.shape)

display(benchmark_results.fillna("-"))

Export terminé : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\docs\benchmark\results_benchmark\benchmark_results.csv
Dimensions : (17, 16)


,solution,horizon_days,model,evaluation_set,evaluation_status,mae,rmse,r2,training_time_seconds,inference_time_seconds,inference_time_per_row_ms,peak_cpu_percent,peak_ram_mb,software_license_cost_eur,selection_decision,notes
0,PyCaret,7.0,HuberRegressor,test,POC réalisé,0.008293,0.012347,-0.253132,41.14,0.444908,0.754081,100.0,15276.83,0.0,Non retenu,Entraînement et ressources mesurés globalement...
1,PyCaret,7.0,LassoLars,test,POC réalisé,0.018931,0.020405,-2.422448,41.14,0.380458,0.644844,100.0,15276.83,0.0,Non retenu,Entraînement et ressources mesurés globalement...
2,PyCaret,7.0,DummyRegressor,test,POC réalisé,0.018931,0.020405,-2.422448,41.14,0.15692,0.265965,100.0,15276.83,0.0,Non retenu,Entraînement et ressources mesurés globalement...
3,PyCaret,30.0,Lasso,test,POC réalisé,0.019914,0.021169,-7.029626,36.94,0.386425,0.654957,100.0,15159.97,0.0,Non retenu,Entraînement et ressources mesurés globalement...
4,PyCaret,30.0,LassoLars,test,POC réalisé,0.019914,0.021169,-7.029626,36.94,0.463464,0.785532,100.0,15159.97,0.0,Non retenu,Entraînement et ressources mesurés globalement...
5,PyCaret,30.0,DummyRegressor,test,POC réalisé,0.019914,0.021169,-7.029626,36.94,0.179289,0.303879,100.0,15159.97,0.0,Non retenu,Entraînement et ressources mesurés globalement...
6,AutoGluon,7.0,WeightedEnsemble_L2,test,POC réalisé,0.008544,0.011058,-0.005152,89.37,0.159108,0.269675,100.0,15156.62,0.0,Retenu pour validation applicative,Entraînement et ressources mesurés globalement...
7,AutoGluon,7.0,NeuralNetTorch,test,POC réalisé,0.00782,0.011156,-0.023035,89.37,0.254139,0.430743,100.0,15156.62,0.0,Retenu pour validation applicative,Entraînement et ressources mesurés globalement...
8,AutoGluon,7.0,LightGBMXT,test,POC réalisé,0.013604,0.015526,-0.981313,89.37,0.018693,0.031683,100.0,15156.62,0.0,Retenu pour validation applicative,Entraînement et ressources mesurés globalement...
9,AutoGluon,30.0,WeightedEnsemble_L2,test,POC réalisé,0.006251,0.007711,-0.065475,44.1,0.078105,0.132382,100.0,15354.15,0.0,Retenu pour validation applicative,Entraînement et ressources mesurés globalement...


## Exporter les resultats du benchmark

In [7]:
export_path = results_dir / "benchmark_results.csv"

benchmark_results.to_csv(export_path, index=False)

print("Export terminé :", export_path)

Export terminé : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\docs\benchmark\results_benchmark\benchmark_results.csv
